In [2]:
import xmlrpc.client # Comunicar con Odoo 
import pandas as pd 
import base64 # Transformar Texto a Base64 y viceversa
import io  # Trabajar datos en memoria como si fuera archivos sin guardar en disco
import numpy as np
import os
from dotenv import load_dotenv

### Fuentes de Información

In [3]:
# Se agrega la variable para modificar la ruta de guardado
ruta_carpeta = "C:/Users/LENOVO/Documents/Git-Proyecto-Tranvial-Tatsa/información_Proyecto_BI/"
df_kardex_anterior = pd.read_parquet(f"{ruta_carpeta}Kardex_Unificado_Anterior.parquet") #En Parquet para mayor rapidez 
df_llaves_original = pd.read_excel("Bd_Llaves_Actualizado.xlsx")

### Credenciales de Ingreso 

In [4]:
load_dotenv() # Cargar las variables de entorno desde el archivo .env
url =  os.getenv("URL_ODOO")
username =  os.getenv("USER_JIM")
password = os.getenv("PASSWORD_JIM")
db = os.getenv("BD_ODOO")

### Objecto de Autentificar y Manipulación

In [5]:
"""
    1. CONECTAR A ODOO - SE CREA OBJECTO PARA IDENTIFICAR + OBJECTO PARA HACER MODIFICACIONES
"""

# Crear objecto - Para conectar
common = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/common")

# Identificarme
uid = common.authenticate(db, username, password, {})
print(f"uid conectado : {uid}")

# Crear objecto  - Para poder hacer consultas al modelo

models = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/object", allow_none=True)

uid conectado : 20


### Dim_Producto

In [6]:
def limpando_producto(modelo_extraido, llave):
    # Para limpiar los diccionarios con ID + Valor solo quedarnos con el valor
    df = []

    for diccionario in modelo_extraido:
        nuevo_diccionario = {}

        for clave, valor in diccionario.items():
            if isinstance(valor, list) and len(valor) == 2:
                nuevo_diccionario[clave] = valor[1]

            elif valor is False:
                nuevo_diccionario[clave] = None

            else:
                nuevo_diccionario[clave] = valor

        df.append(nuevo_diccionario)

    # Limpiando el DF 
    df = pd.DataFrame(df)
    df[["Familia", "Familia_2", "Familia_3"]] = df["categ_id"].str.split("/", n=2, expand=True)
    df = df.rename(
        columns={
            "name" : "Producto", "numero_parte" : "Nro_Parte", "product_brand_id" : "Marca", 
            "default_code" : "Cod_Producto", "uom_id": "Unidad_Medida", "tipo_uso": "Uso"
            }
        )

    df =  df.drop(columns=["categ_id"])
    df = df[["Cod_Producto", "Producto", "Nro_Parte", "Marca", "Unidad_Medida", "Familia", "Familia_2", "Familia_3", "Uso"]]

    cod_proyecto = ["G2020382","G2020383","G2011309","G2011310","G2290040","G2290039","G2290041","G2290038","G2290037","G2290036",
                    "G2011308","G2011311","G2230040","G2020385","G2210081","G2020384","G2210079","G2210080","G2210078","G2230041","G2230042"]
    condiciones = [
        df["Cod_Producto"].isin(cod_proyecto),
        df["Producto"].str.contains("TA0", case=False, na=False)
    ]

    resultado = ["Proyecto Chasis", "Proyecto Chasis"]

    df["Tipo_Proyecto"] = np.select(
        condiciones, resultado, default="General"
    )
    df['Familia'] = df['Familia'].str.strip()

    df = pd.merge(
        df,
        llave[["Cod_Producto", "Cod_Grupo_Sku", "Producto_Grupo"]],
        on= "Cod_Producto",
        how= "left"
    )

    df["Cod_Grupo_Sku"] = df["Cod_Grupo_Sku"].fillna(df["Cod_Producto"])
    df["Producto_Grupo"] = df["Producto_Grupo"].fillna(df["Producto"])
    
    return df 

In [7]:
"""
    1. CONEXION A DF 
"""
productos = models.execute_kw(
    db, uid, password,
    "product.template",
    "search_read",
    [
        [
            ["default_code", "!=", False]
        ]
    ],
    {
        "fields" : ["name", "numero_parte", "product_brand_id", "default_code",
                    "categ_id", "uom_id", "tipo_uso"],
        "context" :{
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2],
            "search_default_consumable": 1
        }
    }
)

"""
    2. LIMPIEZA DF
"""
df_productos = limpando_producto(productos, df_llaves_original)

In [8]:
# 1. Hacer una copia de trabajo
df_export_producto = df_productos.copy()

# 2. Convertir cualquier columna de tipo 'period' a string (o datetime)
for col in df_export_producto.columns:
    if isinstance(df_export_producto[col].dtype, pd.PeriodDtype):
        df_export_producto[col] = df_export_producto[col].astype(str)

# 3. Asegurar que las columnas de tipo 'str' genéricas no causen conflicto
str_cols = df_export_producto.select_dtypes(include=['str', 'string']).columns
df_export_producto[str_cols] = df_export_producto[str_cols].astype(object)

# 4. Exportar a Parquet sin reiniciar el kernel ni registrar extensiones duplicadas
df_export_producto.to_parquet(
    f"{ruta_carpeta}Dim_Producto.parquet",
    index=False,
    engine="pyarrow",
    coerce_timestamps="ms",
    allow_truncated_timestamps=True
)

### Dim_Flota

In [9]:
def limpieza_flota(modelo_flota):
    df = []

    for diccionario in modelo_flota:
        nuevo_diccionario = {}
        
        for clave, valor in diccionario.items():
            if isinstance(valor, list) and len(valor) == 2:
                nuevo_diccionario[clave] = valor[1]
            
            elif valor is False:
                nuevo_diccionario[clave] = None
            
            else:
                nuevo_diccionario[clave] = valor

        df.append(nuevo_diccionario)

    df = pd.DataFrame(df)
    df["code"] = df["code"].fillna(df["license_plate"])
    df = df[df["license_plate"].notna()]

    condiciones = [
        df["code"].str.startswith("CHA", na=False) | df["code"].str.startswith("9BM", na=False),
        df["type_id"].isna()
    ]

    resultado = [
        "CHASIS/MERCEDES",
        "OTROS"
    ]

    df["Tamaño_Bus"] = np.select(condiciones, resultado, default=df["type_id"])
    df["Tamaño_Bus"] = df["Tamaño_Bus"].str.upper()
    df = df.rename(columns={"code" : "Cod_Bus"})
    df = df[["Cod_Bus", "Tamaño_Bus"]]
    df = df.drop_duplicates()
    
    return df

In [10]:
"""
    1. CONEXION A DF 
"""
flota = models.execute_kw(
    db, uid, password,
    "fleet.vehicle",
    "search_read",
    [
        [
           # ["default_code", "!=", False]
        ]
    ],
    {
        "fields" : ["license_plate", "type_id", "code"],
        "context" :{
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2]
        }
    }
)

df_flota = limpieza_flota(flota)
#df_flota.to_excel("Dim_Flota.xlsx", index=False)

In [11]:
# 1. Hacer una copia de trabajo
df_export_flota = df_flota.copy()

# 2. Convertir cualquier columna de tipo 'period' a string (o datetime)
for col in df_export_flota.columns:
    if isinstance(df_export_flota[col].dtype, pd.PeriodDtype):
        df_export_flota[col] = df_export_flota[col].astype(str)

# 3. Asegurar que las columnas de tipo 'str' genéricas no causen conflicto
str_cols = df_export_flota.select_dtypes(include=['str', 'string']).columns
df_export_flota[str_cols] = df_export_flota[str_cols].astype(object)

# 4. Exportar a Parquet sin reiniciar el kernel ni registrar extensiones duplicadas
df_export_flota.to_parquet(
    f"{ruta_carpeta}Dim_Flota.parquet",
    index=False,
    engine="pyarrow",
    coerce_timestamps="ms",
    allow_truncated_timestamps=True
)

### Requerimiento Clasificacion OC

#### Funcion para identificar los periodos del Requerimiento

In [12]:
"""
    1. PARA EXTRAER EL ID DEL PERIODO DE WIZARD REPORTE DE INVENTARIO
"""

periodos = models.execute_kw(
    db, uid, password,
    "account.period.kardex",
    "search_read",
    [[]],
    {   
        "fields":["id", "code", "display_name"],
        "context": {
            "lang": "es_PE", 
            "tz": "America/Lima",
            "allowed_company_ids": [2]
        }
    }
)

df_periodo_reporte_inv = pd.DataFrame(periodos) #El id es necesario 
df_periodo_reporte_inv[df_periodo_reporte_inv["code"] == "202612"]

,id,code,display_name
55,62,202612,202612


#### Requerimiento Tatsa 2026

In [13]:
"""
    2. GENERAR EL WIZARD CON EL FILTRO PERIDO
"""

wizard_id_reporte = models.execute_kw(
    db, uid, password,
    "report.futuro.use",
    "create",
    [
        {
            "fecha_inicio": 51, # Periodo 202501
            "fecha_final" : 62, # Periodo 202512
            "product_ids": [(6, 0, [])], # Sin filtro Repuestos
        }
    ],
    {
        "context": {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2]
        }
    }
)

print("Wizard creado:", wizard_id_reporte)

"""
    3. EJECUTAR EL WIZARD CREADO
"""
accion_wizard_reporte = models.execute_kw(
    db, uid, password,
    "report.futuro.use",
    "descargar_report",
    [
        [wizard_id_reporte]
    ],
    {
        "context":{
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2]
        }
    }
)


popup_id = accion_wizard_reporte["res_id"]
print(f"Id creado del wizard {popup_id}")

"""
    4. LEER Y EJECUTAR EL WIZARD Y EXTRAER EL DATAFRAME
"""
popup = models.execute_kw(
    db, uid, password,
    "popup.it",
    "read",
    [[popup_id]],
    {
        "fields" : ["output_name", "output_file", "display_name"],
        "context" : {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2]
        }
    }

)[0]

print("\nInformacion Extra")

print(popup["output_name"])                  # nombre del archivo: REPORTE DE INVENTARIO.xlsx
print(type(popup["output_file"]))            # str: el Excel llega escrito como texto
print(len(popup["output_file"]))             # tamaño del texto

# 2) Texto base64 -> bytes del archivo (Se manda en texto para no descargar el archivo)
#    Odoo no puede enviar un archivo binario por XML-RPC, así que lo envía como
#    texto (base64). b64decode lo devuelve a su forma original: los bytes del .xlsx.

contenido = base64.b64decode(popup["output_file"])
print(contenido[:2]) # b'PK' -> así empieza todo .xlsx (es un zip)

# 3) Bytes -> "archivo en memoria"
#    pandas espera un archivo. BytesIO hace que esos bytes se comporten como uno,
#    sin crearlo en el disco.
archivo_en_memoria = io.BytesIO(contenido)

# 4) Excel -> tabla
#    dtype=str lee todo como texto, así un código '00123' no se convierte en 123.
df_inv_tat_26 = pd.read_excel(archivo_en_memoria, dtype=str, header=9)

Wizard creado: 2105
Id creado del wizard 92213

Informacion Extra
REPORTE DE INVENTARIO.xlsx
<class 'str'>
1587384
b'PK'


#### Requerimeinto Tvl 2026

In [14]:
"""
    2. GENERAR EL WIZARD CON EL FILTRO PERIDO
"""

wizard_id_reporte = models.execute_kw(
    db, uid, password,
    "report.futuro.use",
    "create",
    [
        {
            "fecha_inicio": 51, # Periodo 202501
            "fecha_final" : 62, # Periodo 202512
            "product_ids": [(6, 0, [])], # Sin filtro Repuestos
        }
    ],
    {
        "context": {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [1]
        }
    }
)

print("Wizard creado:", wizard_id_reporte)

"""
    3. EJECUTAR EL WIZARD CREADO
"""
accion_wizard_reporte = models.execute_kw(
    db, uid, password,
    "report.futuro.use",
    "descargar_report",
    [
        [wizard_id_reporte]
    ],
    {
        "context":{
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [1]
        }
    }
)


popup_id = accion_wizard_reporte["res_id"]
print(f"Id creado del wizard {popup_id}")

"""
    4. LEER Y EJECUTAR EL WIZARD Y EXTRAER EL DATAFRAME
"""
popup = models.execute_kw(
    db, uid, password,
    "popup.it",
    "read",
    [[popup_id]],
    {
        "fields" : ["output_name", "output_file", "display_name"],
        "context" : {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [1]
        }
    }

)[0]

print("\nInformacion Extra")

print(popup["output_name"])                  # nombre del archivo: REPORTE DE INVENTARIO.xlsx
print(type(popup["output_file"]))            # str: el Excel llega escrito como texto
print(len(popup["output_file"]))             # tamaño del texto

# 2) Texto base64 -> bytes del archivo (Se manda en texto para no descargar el archivo)
#    Odoo no puede enviar un archivo binario por XML-RPC, así que lo envía como
#    texto (base64). b64decode lo devuelve a su forma original: los bytes del .xlsx.

contenido = base64.b64decode(popup["output_file"])
print(contenido[:2]) # b'PK' -> así empieza todo .xlsx (es un zip)

# 3) Bytes -> "archivo en memoria"
#    pandas espera un archivo. BytesIO hace que esos bytes se comporten como uno,
#    sin crearlo en el disco.
archivo_en_memoria = io.BytesIO(contenido)

# 4) Excel -> tabla
#    dtype=str lee todo como texto, así un código '00123' no se convierte en 123.
df_inv_tvl_26 = pd.read_excel(archivo_en_memoria, dtype=str, header=9)

Wizard creado: 2106
Id creado del wizard 92214

Informacion Extra
REPORTE DE INVENTARIO.xlsx
<class 'str'>
718352
b'PK'


#### Unificar la clasificacion de la OC

In [15]:
def unificar_clasificacio_OC(rq, empresa):
    df = rq.copy()
    df = df[["CENTRO DE COSTO", "ORDEN DE COMPRA", "PROVEEDOR ELEGIDO"]]
    df = df[df["ORDEN DE COMPRA"].notna()]
    df = df.drop_duplicates()
    df["Empresa"] = empresa
    df = df.rename(
        columns={
            "CENTRO DE COSTO" : "Cuenta Analitica", 
            "ORDEN DE COMPRA": "Cod_Orden_Compra",
            "PROVEEDOR ELEGIDO" : "Proveedor"
            })
    return df

df_clasificacion_rq = pd.concat(
    [
        unificar_clasificacio_OC(df_inv_tat_26, "Tatsa"),
        unificar_clasificacio_OC(df_inv_tvl_26, "Tvl")
    ]
)

### Fact_Kardex

#### OTS New

In [16]:
report_ots_New_tatsa =  models.execute_kw(
    db, uid, password,
    "maintenance.request",
    "search_read",
    [
        [
            #["kardex_date", "!=", False]
        ]
    ],
    {
        "fields": ["name", "maintenance_type"],
        "context": {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2],
        },
        #"limit": 20,
    }
)

df_report_ots_New_tatsa = pd.DataFrame(report_ots_New_tatsa)
df_report_ots_New_tatsa = df_report_ots_New_tatsa[["name", "maintenance_type"]]
df_report_ots_New_tatsa = df_report_ots_New_tatsa.rename(columns={"name": "Cod_Ot", "maintenance_type" : "Tipo_Mantto"})
df_report_ots_New_tatsa["Tipo_Mantto"] = df_report_ots_New_tatsa["Tipo_Mantto"].replace(
    {
        "corrective" : "Correctivo",
        "preventive" : "Preventivo",
        "predictive" : "Preventivo"
    }
)

#### Ots Anterior Tatsa

In [17]:
report_ots_tatsa =  models.execute_kw(
    db, uid, password,
    "orden.trabajo",
    "search_read",
    [
        [
            #["kardex_date", "!=", False]
        ]
    ],
    {
        "fields": ["name", "state", "tipo_mantenimiento", "flota"],
        "context": {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2],
        },
        #"limit": 20,
    }
)

# Limpiar las lista en columnas 
df = []

for diccionario in report_ots_tatsa:
    nuevo_diccionario = {}
    
    for clave, valor in diccionario.items():
        if isinstance(valor, list) and len(valor) == 2:
            nuevo_diccionario[clave] = valor[1]
        
        elif valor is False:
            nuevo_diccionario[clave] = None
        
        else:
            nuevo_diccionario[clave] = valor

    df.append(nuevo_diccionario)

df_report_ots_tatsa = pd.DataFrame(df)
df_report_ots_tatsa = df_report_ots_tatsa[df_report_ots_tatsa["name"].notna()]
df_report_ots_tatsa = df_report_ots_tatsa[["name", "tipo_mantenimiento", "flota"]]
df_report_ots_tatsa["tipo_mantenimiento"] = df_report_ots_tatsa["tipo_mantenimiento"].replace(
    {
        "correctivo" : "Correctivo",
        "correctivo_reactivo" : "Correctivo",
        "correctivo_programado" : "Correctivo",
        "preventivo_programado" : "Preventivo",
        "preventivo" : "Preventivo",
        "predictivo" : "Preventivo", 
        "preventivo_linea": "Preventivo",
        "predictivo_rutina" : "Preventivo",
        False : np.nan
    }
)
df_report_ots_tatsa = df_report_ots_tatsa.rename(
    columns={
        "name" : "Cod_Ot", 
        "tipo_mantenimiento" : "Tipo_Mantto",
        "flota" : "Ot_Flota"
    }
)

df_report_ots_tatsa["Ot_Flota"] = df_report_ots_tatsa["Ot_Flota"].str.extract(r'\((.*?)\)')

#### Ots Anterior Transvial

In [18]:
report_ots_tvl =  models.execute_kw(
    db, uid, password,
    "orden.trabajo",
    "search_read",
    [
        [
            #["kardex_date", "!=", False]
        ]
    ],
    {
        "fields": ["name", "state", "tipo_mantenimiento", "flota"],
        "context": {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [1],
        },
        #"limit": 20,
    }
)

# Para extraer la libreria 

df = []

for diccionario in report_ots_tvl:
    nuevo_diccionario = {}
    
    for clave, valor in diccionario.items():
        if isinstance(valor, list) and len(valor) == 2:
            nuevo_diccionario[clave] = valor[1]
        
        elif valor is False:
            nuevo_diccionario[clave] = None
        
        else:
            nuevo_diccionario[clave] = valor

    df.append(nuevo_diccionario)


df_report_ots_tvl = pd.DataFrame(df)
df_report_ots_tvl = df_report_ots_tvl[df_report_ots_tvl["name"].notna()]
df_report_ots_tvl = df_report_ots_tvl[["name", "tipo_mantenimiento", "flota"]]
df_report_ots_tvl["tipo_mantenimiento"] = df_report_ots_tvl["tipo_mantenimiento"].replace(
    {
        "correctivo" : "Correctivo",
        "correctivo_reactivo" : "Correctivo",
        "correctivo_programado" : "Correctivo",
        "preventivo_programado" : "Preventivo",
        "preventivo" : "Preventivo",
        "predictivo" : "Preventivo", 
        "preventivo_linea": "Preventivo",
        "predictivo_rutina" : "Preventivo",
        False : np.nan
    }
)
df_report_ots_tvl = df_report_ots_tvl.rename(
    columns={
        "name" : "Cod_Ot", 
        "tipo_mantenimiento" : "Tipo_Mantto",
        "flota" : "Ot_Flota"
    }
)
df_report_ots_tvl["Ot_Flota"] = df_report_ots_tvl["Ot_Flota"].str.extract(r'\((.*?)\)')

#### Transferencias

In [19]:
#### Funcion de limpieza de transferencia
def limpando_transferencias(modelo_extraido):
    # Para limpiar los diccionarios con ID + Valor solo quedarnos con el valor
    df = []

    for diccionario in modelo_extraido:
        nuevo_diccionario = {}

        for clave, valor in diccionario.items():
            if isinstance(valor, list) and len(valor) == 2:
                nuevo_diccionario[clave] = valor[1]

            elif valor is False:
                nuevo_diccionario[clave] = None

            else:
                nuevo_diccionario[clave] = valor

        df.append(nuevo_diccionario)

    # Limpiando el DF 
    df = pd.DataFrame(df)
    df["flota_name"] = df["flota_name"].str.extract(r'\((.*?)\)')
    df["vehicle_ot_id"] = df["vehicle_ot_id"].str.extract(r'\((.*?)\)')
    df["ots_origenes_name"] = df["ots_origenes_name"].str.extract(r"'(.*?)'")
    df["Cod_Bus"] = df["flota_name"].fillna(df["vehicle_ot_id"])
    df["Cod_Ot"] = df["ots_origenes_name"].fillna(df["request_ot"])
    df = df[["name", "Cod_Bus", "Cod_Ot"]]
    df = df.rename(columns={"name" : "Cod_Albaran"})
    return df 

#### TRANSFERECNIA TATSA

"""
    Extrayendo el modelo Odoo
"""
report_transferencias_tatsa =  models.execute_kw(
    db, uid, password,
    "stock.picking",
    "search_read",
    [
        [
            ["kardex_date", "!=", False],
            ["kardex_date", ">=", "2026-01-01 00:00:00"]
        ]
    ],
    {
        "fields": ["name", "kardex_date", "flota_name", "ots_origenes_name", "request_ot", "vehicle_ot_id", "state"],
        "context": {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2],
        },
        #"limit": 2,
    }
)

"""
    Aplicando funcion de limpieza
"""

df_report_transferencias_tatsa  = limpando_transferencias(report_transferencias_tatsa)
df_report_transferencias_tatsa = pd.merge(
    df_report_transferencias_tatsa,
    pd.concat([df_report_ots_New_tatsa, df_report_ots_tatsa]),
    on="Cod_Ot",
    how="left"

)

df_report_transferencias_tatsa["Cod_Bus"] = df_report_transferencias_tatsa["Cod_Bus"].fillna(df_report_transferencias_tatsa["Ot_Flota"])
df_report_transferencias_tatsa = df_report_transferencias_tatsa.drop(columns=["Ot_Flota"])


#### TRANSFERENCIA TVL

"""
    Extrayendo el modelo Odoo
"""
report_transferencias_tvl =  models.execute_kw(
    db, uid, password,
    "stock.picking",
    "search_read",
    [
        [
            ["kardex_date", "!=", False],
            ["kardex_date", ">=", "2026-01-01 00:00:00"]
        ]
    ],
    {
        "fields": ["name", "kardex_date", "flota_name", "ots_origenes_name", "request_ot", "vehicle_ot_id", "state"],
        "context": {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [1],
        },
        #"limit": 100,
    }
)

"""
    Aplicando funcion de limpieza
"""
df_report_transferencias_tvl  = limpando_transferencias(report_transferencias_tvl)
df_report_transferencias_tvl = pd.merge(
    df_report_transferencias_tvl,
    pd.concat([df_report_ots_New_tatsa, df_report_ots_tvl]),
    on="Cod_Ot",
    how="left"
)
df_report_transferencias_tvl["Cod_Bus"] = df_report_transferencias_tvl["Cod_Bus"].fillna(df_report_transferencias_tvl["Ot_Flota"])
df_report_transferencias_tvl = df_report_transferencias_tvl.drop(columns=["Ot_Flota"])

#### Kardex Tatsa 2026

In [20]:
"""
    1. DATOS DEL KARDEX
"""

FECHA_INICIO = "2026-01-01"      # AAAA-MM-DD
FECHA_FIN = "2026-12-31"

contexto_kardex = {
    "lang": "es_PE",
    "tz": "America/Lima",
    "allowed_company_ids": [2],
    "tipo": "valorado",
    "type_product": "product",
    "form_view_ref": "view_make_kardex_valorado",
}

# Ubicaciones (stock.location) que marca el formulario por defecto
ubicaciones = [186, 48, 47, 115, 49, 45, 46, 22, 198, 204, 210, 179, 18, 19, 20]

"""
    2. GENERAR EL WIZARD CON EL FILTRO DE FECHAS  (el "create" de tu captura)
"""

wizard_id_kardex = models.execute_kw(
    db, uid, password,
    "make.kardex.valorado",
    "create",
    [
        {
            "fini": FECHA_INICIO,                 # fecha inicio que eliges
            "ffin": FECHA_FIN,                    # fecha fin que eliges
            "fecha_ini_mod": FECHA_INICIO,        # copia que rellena la pantalla
            "fecha_fin_mod": FECHA_FIN,           # copia que rellena la pantalla
            "check_fecha": True,                  # casilla "filtrar por fechas"
            "moneda": "pen",                      # soles
            "alllocations": True,                 # todas las ubicaciones
            "location_ids": [(6, 0, ubicaciones)],# (6,0,lista) = "usa exactamente esta lista"
            "allproducts": True,                  # todos los productos
            "products_ids": [(6, 0, [])],         # lista vacía = sin filtro de productos
            "analizador": False,
            "check_account": False,
        }
    ],
    {"context": contexto_kardex}
)

print("Wizard creado:", wizard_id_kardex)

"""
    3. EJECUTAR EL WIZARD  (el call_button "do_csvtoexcel_transvial")
"""

respuesta_kardex = models.execute_kw(
    db, uid, password,
    "make.kardex.valorado",
    "do_csvtoexcel_transvial",
    [
        [wizard_id_kardex]
    ],
    {"context": contexto_kardex}
)

# La respuesta es un diccionario. Para llegar al id del archivo vamos bajando:
#   respuesta_kardex["params"]                  -> diccionario con mensaje y botones
#   respuesta_kardex["params"]["buttons"]       -> lista de botones (la "caja")
#   respuesta_kardex["params"]["buttons"][0]    -> el primer botón (la "ficha")
#   respuesta_kardex["params"]["buttons"][0]["id"] -> id del archivo adjunto
print(str(respuesta_kardex["params"]["message"]).replace("<br/>", " | "))

adjunto_id = respuesta_kardex["params"]["buttons"][0]["id"]
print("Id del adjunto:", adjunto_id)

"""
    4. LEER EL ADJUNTO  (el Excel está guardado en ir.attachment, campo "datas")
"""

adjunto = models.execute_kw(
    db, uid, password,
    "ir.attachment",
    "read",
    [[adjunto_id]],
    {
        "fields": ["name", "datas"],
        "context": {"lang": "es_PE", "tz": "America/Lima", "allowed_company_ids": [2]},
    }
)[0]                                        # [0]: sacar la ficha de la caja

print(adjunto["name"])
print(len(adjunto["datas"]))

"""
    5. DE TEXTO A TABLA (en memoria, sin descargar)
"""

contenido_kardex = base64.b64decode(adjunto["datas"])     # texto base64 -> bytes del .xlsx
print(contenido_kardex[:2])                               # b'PK'

archivo_en_memoria = io.BytesIO(contenido_kardex)         # bytes -> "archivo" en memoria

# header=None: leemos TODO el Excel sin decidir cuál fila es el encabezado,
# para ver primero cuántas filas de títulos trae arriba.
df_kardex_tatsa_2026 = pd.read_excel(archivo_en_memoria, dtype=str, header=8)
df_kardex_tatsa_2026["Empresa"] = "Tatsa"

Wizard creado: 8156
Se proceso de 2026-01-01 al 2026-12-31. | Lineas procesadas: 72028 | Tiempo: 0 minutos 18 segundos
Id del adjunto: 289963
Kardex Valorado.xlsx
15524324
b'PK'


#### Kardex Tvl 2026

In [21]:
"""
    1. DATOS DEL KARDEX (lo que vas a cambiar)
"""

FECHA_INICIO = "2026-01-01"      # AAAA-MM-DD
FECHA_FIN = "2026-12-31"

# Contexto del Kardex: lo mismo que ves en el Network, con 3 claves extra
contexto_kardex = {
    "lang": "es_PE",
    "tz": "America/Lima",
    "allowed_company_ids": [1],
    "tipo": "valorado",
    "type_product": "product",
    "form_view_ref": "view_make_kardex_valorado",
}

# Ubicaciones (stock.location) que marca el formulario por defecto
ubicaciones = [39, 40, 44, 42, 41, 34, 192, 28, 8, 14, 15, 16]


"""
    2. GENERAR EL WIZARD CON EL FILTRO DE FECHAS  (el "create" de tu captura)
"""

wizard_id_kardex = models.execute_kw(
    db, uid, password,
    "make.kardex.valorado",
    "create",
    [
        {
            "fini": FECHA_INICIO,                 # fecha inicio que eliges
            "ffin": FECHA_FIN,                    # fecha fin que eliges
            "fecha_ini_mod": FECHA_INICIO,        # copia que rellena la pantalla
            "fecha_fin_mod": FECHA_FIN,           # copia que rellena la pantalla
            "check_fecha": True,                  # casilla "filtrar por fechas"
            "moneda": "pen",                      # soles
            "alllocations": True,                 # todas las ubicaciones
            "location_ids": [(6, 0, ubicaciones)],# (6,0,lista) = "usa exactamente esta lista"
            "allproducts": True,                  # todos los productos
            "products_ids": [(6, 0, [])],         # lista vacía = sin filtro de productos
            "analizador": False,
            "check_account": False,
        }
    ],
    {"context": contexto_kardex}
)

print("Wizard creado:", wizard_id_kardex)


"""
    3. EJECUTAR EL WIZARD  (el call_button "do_csvtoexcel_transvial")
"""

respuesta_kardex = models.execute_kw(
    db, uid, password,
    "make.kardex.valorado",
    "do_csvtoexcel_transvial",
    [
        [wizard_id_kardex]
    ],
    {"context": contexto_kardex}
)

# La respuesta es un diccionario. Para llegar al id del archivo vamos bajando:
#   respuesta_kardex["params"]                  -> diccionario con mensaje y botones
#   respuesta_kardex["params"]["buttons"]       -> lista de botones (la "caja")
#   respuesta_kardex["params"]["buttons"][0]    -> el primer botón (la "ficha")
#   respuesta_kardex["params"]["buttons"][0]["id"] -> id del archivo adjunto
print(str(respuesta_kardex["params"]["message"]).replace("<br/>", " | "))

adjunto_id = respuesta_kardex["params"]["buttons"][0]["id"]
print("Id del adjunto:", adjunto_id)       # es el 280160 de tu captura (get_download_ls)


"""
    4. LEER EL ADJUNTO  (el Excel está guardado en ir.attachment, campo "datas")
"""

adjunto = models.execute_kw(
    db, uid, password,
    "ir.attachment",
    "read",
    [[adjunto_id]],
    {
        "fields": ["name", "datas"],
        "context": {"lang": "es_PE", "tz": "America/Lima", "allowed_company_ids": [1]},
    }
)[0]                                        # [0]: sacar la ficha de la caja

print(adjunto["name"])
print(len(adjunto["datas"]))


"""
    5. DE TEXTO A TABLA (en memoria, sin descargar)
"""

contenido_kardex = base64.b64decode(adjunto["datas"])     # texto base64 -> bytes del .xlsx
print(contenido_kardex[:2])                               # b'PK'

archivo_en_memoria = io.BytesIO(contenido_kardex)         # bytes -> "archivo" en memoria

# header=None: leemos TODO el Excel sin decidir cuál fila es el encabezado,
# para ver primero cuántas filas de títulos trae arriba.
df_kardex_tvl_2026 = pd.read_excel(archivo_en_memoria, dtype=str, header=8)
df_kardex_tvl_2026["Empresa"] = "Tvl"

Wizard creado: 8157
Se proceso de 2026-01-01 al 2026-12-31. | Lineas procesadas: 17229 | Tiempo: 0 minutos 4 segundos
Id del adjunto: 289964
Kardex Valorado.xlsx
3547732
b'PK'


#### Union de df

In [22]:
def unificar_kardex(kar_tvl_26, kar_tat_26, transferencia_tvl, transferencia_tat, kardex_anterior, clasificacion_oc):

    # Limpieza TVL
    df_k_tvl = kar_tvl_26.copy()
    df_k_tvl = df_k_tvl[df_k_tvl["Sku"].notna()]
    df_k_tvl = df_k_tvl[
        ["Fecha", "Hora", "Doc. Almacen", "RUC", "Proveedor", "T. OP.", "Sku", "LOTE",
         "Costo Promedio", "Cantidad", "Importe", "Cantidad.1", "Importe.1", "Cantidad.2",
         "Importe.2", "Ubicacion Origen", "Ubicacion Destino", "Almacen", "Pedido", "Empresa"]]
    df_k_tvl = df_k_tvl.rename(columns={"Doc. Almacen" : "Cod_Albaran"})
    df_k_tvl = pd.merge(
        df_k_tvl,
        transferencia_tvl,
        on="Cod_Albaran",
        how="left"
    )

    # Limpieza TAT
    df_k_tat = kar_tat_26.copy()
    df_k_tat = df_k_tat[df_k_tat["Sku"].notna()]
    df_k_tat = df_k_tat[
        ["Fecha", "Hora", "Doc. Almacen", "RUC", "Proveedor", "T. OP.", "Sku", "LOTE",
         "Costo Promedio", "Cantidad", "Importe", "Cantidad.1", "Importe.1", "Cantidad.2",
         "Importe.2", "Ubicacion Origen", "Ubicacion Destino", "Almacen", "Pedido", "Empresa"]]
    df_k_tat = df_k_tat.rename(columns={"Doc. Almacen" : "Cod_Albaran"})
    df_k_tat = pd.merge(
        df_k_tat,
        transferencia_tat,
        on="Cod_Albaran",
        how="left"
    )

    #CONCATENAR

    df_fin = pd.concat([df_k_tvl, df_k_tat], ignore_index=True)
    df_fin["T. OP."] = pd.to_numeric(df_fin["T. OP."], errors="coerce")
    df_fin["Costo Promedio"] = pd.to_numeric(df_fin["Costo Promedio"], errors="coerce")
    df_fin["Cantidad"] = pd.to_numeric(df_fin["Cantidad"], errors="coerce")
    df_fin["Importe"] = pd.to_numeric(df_fin["Importe"], errors="coerce")
    df_fin["Cantidad.1"] = pd.to_numeric(df_fin["Cantidad.1"], errors="coerce")
    df_fin["Importe.1"] = pd.to_numeric(df_fin["Importe.1"], errors="coerce")
    df_fin["Cantidad.2"] = pd.to_numeric(df_fin["Cantidad.2"], errors="coerce")
    df_fin["Importe.2"] = pd.to_numeric(df_fin["Importe.2"], errors="coerce")
    df_fin["Fecha"] = pd.to_datetime(df_fin["Fecha"], errors="coerce")

    # unificar la columna fecha y hora
    df_fin["Fecha_Completa"] = (df_fin["Fecha"].dt.strftime('%Y-%m-%d') + ' ' + df_fin["Hora"].astype(str))

    # convierto la columna fecha completa en tipo de dato:fecha
    df_fin["Fecha_Completa"] = pd.to_datetime(df_fin["Fecha_Completa"], errors="coerce")

    df_fin = df_fin.rename(
        columns={
            "RUC" : "Ruc",
            "T. OP." : "Cod_Movimiento",
            "Sku" : "Cod_Producto",
            "LOTE" : "Lote",
            "Costo Promedio" : "Precio_Unt",
            "Cantidad" : "Cant_Ingreso",
            "Importe" : "Valor_Ingreso",
            "Cantidad.1" : "Cant_Salida",
            "Importe.1" : "Valor_Salida",
            "Cantidad.2" : "Cant_Saldo",
            "Importe.2" : "Valor_Saldo",
            "Ubicacion Origen" : "Ubi_Origen",
            "Ubicacion Destino" : "Ubi_Destino",
            "Pedido" : "Cod_Orden_Compra",
        }
    )

    df_fin = df_fin[
        ["Fecha", "Fecha_Completa", "Cod_Albaran", "Ruc", "Proveedor", "Cod_Movimiento", "Cod_Producto", "Lote", "Precio_Unt",
         "Cant_Ingreso", "Valor_Ingreso", "Cant_Salida", "Valor_Salida", "Cant_Saldo", "Valor_Saldo", "Ubi_Origen", "Ubi_Destino",
         "Almacen", "Cod_Orden_Compra", "Empresa", "Cod_Bus", "Cod_Ot", "Tipo_Mantto"]]

    df_fin = df_fin[df_fin["Cod_Movimiento"].notna()]

    df_fin = pd.merge(
        df_fin,
        clasificacion_oc,
        on=["Cod_Orden_Compra", "Empresa", "Proveedor"],
        how="left"
    )

    df_fin = pd.concat([df_fin, kardex_anterior], ignore_index=True)

    return df_fin


df_kardex_final = unificar_kardex(
    df_kardex_tvl_2026, df_kardex_tatsa_2026,df_report_transferencias_tvl, 
    df_report_transferencias_tatsa, df_kardex_anterior, df_clasificacion_rq)

In [23]:
# 1. Hacer una copia de trabajo
df_export = df_kardex_final.copy()

# 2. Convertir cualquier columna de tipo 'period' a string (o datetime)
for col in df_export.columns:
    if isinstance(df_export[col].dtype, pd.PeriodDtype):
        df_export[col] = df_export[col].astype(str)

# 3. Asegurar que las columnas de tipo 'str' genéricas no causen conflicto
str_cols = df_export.select_dtypes(include=['str', 'string']).columns
df_export[str_cols] = df_export[str_cols].astype(object)

# 4. Exportar a Parquet sin reiniciar el kernel ni registrar extensiones duplicadas
df_export.to_parquet(
    f"{ruta_carpeta}Fact_Kardex.parquet",
    index=False,
    engine="pyarrow",
    coerce_timestamps="ms",
    allow_truncated_timestamps=True
)

### Fact_Kilomentraje

In [24]:
"""
    1. CONEXION A DF KILOMETRAJE
"""

kilometraje = models.execute_kw(
    db, uid, password,
    "maintenance.kilometer.history",
    "search_read",
    [[]],
    {
        "fields" : ["name", "date", "km_inicial", "km_recorridos", "dif_km"],
        "context" : {
            "lang" : "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2],
        }
    }
)

df = []

for diccionario in kilometraje:
    nuevo_diccionario = {}

    for clave, valor in diccionario.items():
        if isinstance(valor, list) and len(valor) == 2:
            nuevo_diccionario[clave] = valor[1]

        elif valor is False:
            nuevo_diccionario[clave] = None

        else:
            nuevo_diccionario[clave] = valor

    df.append(nuevo_diccionario)


df_kilometraje = pd.DataFrame(df)
df_kilometraje = df_kilometraje.rename(
    columns={
        "name" : "Cod_Bus", 
        "date": "Fecha_Km", 
        "km_inicial" : "Km_Ini",
        "km_recorridos" : "Km_Fin",
        "dif_km" : "Km_Diario"
    }
)
df_kilometraje = df_kilometraje[
    ["Cod_Bus", "Fecha_Km", "Km_Ini", "Km_Fin", "Km_Diario"]
]
df_kilometraje["Fecha_Km"] = pd.to_datetime(df_kilometraje["Fecha_Km"], errors="coerce")

str_cols = df_kilometraje.select_dtypes(include=["str", "string"]).columns
df_kilometraje[str_cols] = df_kilometraje[str_cols].astype(object)

df_kilometraje.to_parquet(
    f"{ruta_carpeta}Fact_Km.parquet",
    index = False,
    engine = "pyarrow",
    coerce_timestamps = "ms",
    allow_truncated_timestamps = True 
)